In [1]:
import pandas as pd
import sys
import os

current_dir = os.getcwd()
parent_dir = os.path.join(current_dir, '..')
sys.path.append(parent_dir)

from data_processing import split_chronologically

# Explanation of chronological split function

In [ ]:
"""
Explain the chronological split with concrete examples
"""
print("=" * 60)
print("HOW THE CHRONOLOGICAL SPLIT WORKS")
print("=" * 60)

# Get the split result
result = split_chronologically(data_path='~/cmu/goalsetting-recommendation-algorithm/time-series-predictor/data/data_tidied.csv', K=3)

train_data = result['train_data']
test_data = result['test_data']
K = result['metadata']['K']

print(f"\n📋 SPLIT STRATEGY:")
print(f"• For each student individually:")
print(f"  - Take their complete time series (all weeks)")
print(f"  - Last {K} weeks → TEST SET")
print(f"  - Everything before → TRAINING SET")
print(f"• This ensures chronological order: we never use future data to predict the past")
print(f"• No temporal leakage: model only sees past to predict future")

# Load original data to show complete picture
original_data = pd.read_csv('~/cmu/goalsetting-recommendation-algorithm/time-series-predictor/data/data_tidied.csv')

# Show examples for 4 students
students_to_show = original_data['name'].unique()[:4]

print(f"\n" + "=" * 60)
print("CONCRETE EXAMPLES")
print("=" * 60)

for i, student in enumerate(students_to_show):
    print(f"\n--- STUDENT {i+1}: {student[:]} ---")
    
    # Get all data for this student
    student_all = original_data[original_data['name'] == student].sort_values('week')
    student_train = train_data[train_data['name'] == student].sort_values('week')
    student_test = test_data[test_data['name'] == student].sort_values('week')
    
    total_weeks = len(student_all)
    
    print(f"Complete timeline ({total_weeks} weeks):")
    weeks_str = " ".join([f"W{w:2d}" for w in student_all['week']])
    perf_str = " ".join([f"{p:3.0f}" for p in student_all['proficient']])
    print(f"  Weeks: {weeks_str}")
    print(f"  Skills: {perf_str}")
    
    if len(student_train) > 0 and len(student_test) > 0:
        print(f"\nAfter split:")
        print(f"  TRAINING → Weeks {student_train['week'].min():2d} to {student_train['week'].max():2d} ({len(student_train)} weeks)")
        print(f"  TEST     → Weeks {student_test['week'].min():2d} to {student_test['week'].max():2d} ({len(student_test)} weeks)")
        print(f"  Split point: {student_train['week'].max()} | {student_test['week'].min()}")
    else:
        print(f"  ⚠️  Insufficient data for split (≤{K} weeks total)")
    
    # Show visual representation
    if len(student_train) > 0 and len(student_test) > 0:
        visual = ""
        for week in student_all['week']:
            if week in student_train['week'].values:
                visual += "T "
            elif week in student_test['week'].values:
                visual += "E "
            else:
                visual += "? "
        print(f"  Visual:   {visual}(T=Train, E=tEst)")

print(f"\n" + "=" * 60)
print("WHY THIS SPLIT MAKES SENSE")
print("=" * 60)

print("✅ ADVANTAGES:")
print("  1. REALISTIC: Model sees past → predicts future (like real usage)")
print("  2. NO LEAKAGE: Future data never contaminates training")
print("  3. FAIR COMPARISON: All students get same test period length")
print("  4. SUFFICIENT TRAINING: Most students get 7-8 weeks for training")

print(f"\n📊 SPLIT STATISTICS:")
print(f"  • Total students: {result['metadata']['total_students']}")
print(f"  • Students processed: {result['metadata']['students_processed']}")
print(f"  • Students with insufficient data: {result['metadata']['students_insufficient_data']}")
print(f"  • Training samples: {result['metadata']['train_samples']}")
print(f"  • Test samples: {result['metadata']['test_samples']}")

HOW THE CHRONOLOGICAL SPLIT WORKS
Use create_time_series_splits() for proper time series cross-validation.
STEP 3: Split chronologically
Loading tidied data from: ~/cmu/goalsetting-recommendation-algorithm/time-series-predictor/data/data_tidied.csv
Loaded data shape: (1893, 3)



NameError: name 'analyze_data_for_split' is not defined

# Demo of the chronological split function

In [2]:

print("Testing chronological split function...")
print("=" * 50)

# Execute the split
result = split_chronologically(data_path='~/cmu/goalsetting-recommendation-algorithm/time-series-predictor/data/data_tidied.csv')

# Extract the components
train_data = result['train_data']
test_data = result['test_data']
split_info = result['split_info']
metadata = result['metadata']

print("\n" + "=" * 50)
print("DETAILED ANALYSIS OF SPLIT RESULTS")
print("=" * 50)

# Analyze training data
print("\n📊 TRAINING DATA ANALYSIS:")
print(f"Shape: {train_data.shape}")
print(f"Students: {train_data['name'].nunique()}")
print(f"Week range: {train_data['week'].min()} to {train_data['week'].max()}")
print(f"Proficient scores range: {train_data['proficient'].min():.1f} to {train_data['proficient'].max():.1f}")
print(f"Sample of training data:")
print(train_data.head())

# Analyze test data
print("\n📊 TEST DATA ANALYSIS:")
print(f"Shape: {test_data.shape}")
print(f"Students: {test_data['name'].nunique()}")
print(f"Week range: {test_data['week'].min()} to {test_data['week'].max()}")
print(f"Proficient scores range: {test_data['proficient'].min():.1f} to {test_data['proficient'].max():.1f}")
print(f"Sample of test data:")
print(test_data.head())

# Analyze split info
print("\n📊 SPLIT INFO ANALYSIS:")
print(f"Number of students: {len(split_info)}")
print(f"Training weeks per student:")
print(split_info['train_weeks'].describe())
print(f"Sample split info:")
print(split_info.head())

# Verify data integrity
print("\n✅ DATA INTEGRITY CHECKS:")

# Check 1: No overlap between train and test weeks for any student
overlap_issues = []
for _, row in split_info.iterrows():
    student = row['student']
    train_weeks = set(range(row['train_week_range'][0], row['train_week_range'][1] + 1))
    test_weeks = set(range(row['test_week_range'][0], row['test_week_range'][1] + 1))
    
    if train_weeks.intersection(test_weeks):
        overlap_issues.append(student)

print(f"Students with train/test week overlap: {len(overlap_issues)}")

# Check 2: All students should have exactly K=3 test weeks
test_week_counts = test_data.groupby('name').size()
students_wrong_test_count = test_week_counts[test_week_counts != metadata['K']]
print(f"Students with wrong test week count: {len(students_wrong_test_count)}")

# Check 3: Total samples should match
total_samples = len(train_data) + len(test_data)
expected_samples = metadata['original_samples'] - metadata['students_insufficient_data'] * metadata['K']
print(f"Sample count verification: {total_samples} vs expected ~{expected_samples}")

# Example of how to use for modeling
print("\n🔧 EXAMPLE USAGE FOR MODELING:")
print("# Extract data for modeling")
print("train_X = train_data[['week']]  # Features")
print("train_y = train_data['proficient']  # Target")
print("test_X = test_data[['week']]")
print("test_y = test_data['proficient']")
print("\n# For time-series AR model, you'd create lagged features:")

# Demonstrate creating lagged features for one student
sample_student = train_data['name'].iloc[0]
student_train_data = train_data[train_data['name'] == sample_student].sort_values('week')

print(f"\nExample for student {sample_student}:")
print("Original data:")
print(student_train_data[['week', 'proficient']])

# Create simple lag features
student_train_data_lag = student_train_data.copy()
student_train_data_lag['proficient_lag1'] = student_train_data_lag['proficient'].shift(1)
student_train_data_lag['proficient_lag2'] = student_train_data_lag['proficient'].shift(2)

print("\nWith lag features (for AR model):")
print(student_train_data_lag[['week', 'proficient', 'proficient_lag1', 'proficient_lag2']].dropna())

print("\n" + "=" * 50)
print("Split result is now available as 'result' dictionary")
print("Key components:")
for key in result.keys():
    if key in ['train_data', 'test_data', 'split_info']:
        print(f"  result['{key}'] - shape: {result[key].shape}")
    else:
        print(f"  result['{key}'] - {type(result[key])}") 

Testing chronological split function...
STEP 3: Split chronologically
Loading tidied data from: ~/cmu/goalsetting-recommendation-algorithm/time-series-predictor/data/data_tidied.csv
Loaded data shape: (1893, 3)

Analyzing data structure for chronological split...
Data analysis for split determination:
  Total students: 184
  Week range: -2 to 8
  Weeks per student - median: 11.0, range: 3-11
  Recommended K: 3
  Reasoning: With median 11.0 weeks per student, K=3 provides ~27.3% test data while ensuring ≥5 training weeks

Splitting data with K = 3 test weeks per student...
Chronological split complete!
  Students processed: 183
  Students with insufficient data (≤3 weeks): 1
  Training samples: 1341
  Test samples: 549
  Training weeks per student: count    183.000000
mean       7.327869
std        1.153894
min        1.000000
25%        7.000000
50%        8.000000
75%        8.000000
max        8.000000
dtype: float64
  Test weeks per student: count    183.0
mean       3.0
std        

## Answer of why test week has large range [2,8]: because some students dropout at week 4, thus their test weeks (2,3,4) contains 2

In [4]:
def analyze_test_week_range():
    print("=== WHY TEST WEEKS RANGE FROM 2 TO 8 ===")
    print()
    
    # Get the split data
    result = split_chronologically(data_path='~/cmu/goalsetting-recommendation-algorithm/time-series-predictor/data/data_tidied.csv')
    test_data = result['test_data']
    
    # Load original data to see student timelines
    original_data = pd.read_csv('~/cmu/goalsetting-recommendation-algorithm/time-series-predictor/data/data_tidied.csv')
    
    # Analyze when different students end
    student_timelines = original_data.groupby('name')['week'].agg(['min', 'max']).reset_index()
    student_timelines.columns = ['name', 'first_week', 'last_week']
    
    print("🎯 THE KEY INSIGHT:")
    print("Not all students have the same timeline!")
    print()
    
    # Show distribution of ending weeks
    ending_week_counts = student_timelines['last_week'].value_counts().sort_index()
    print("📊 Students by their ENDING week:")
    for week, count in ending_week_counts.items():
        print(f"  {count:3d} students end at week {week}")
    
    print()
    print("💡 Since we take the LAST 3 weeks per student:")
    print("  • Students ending at week 5 → test weeks: 3, 4, 5")
    print("  • Students ending at week 6 → test weeks: 4, 5, 6") 
    print("  • Students ending at week 7 → test weeks: 5, 6, 7")
    print("  • Students ending at week 8 → test weeks: 6, 7, 8")
    
    print()
    print("🔍 CONCRETE EXAMPLES:")
    
    # Show a few examples
    examples = []
    for ending_week in sorted(ending_week_counts.index):
        student_example = student_timelines[student_timelines['last_week'] == ending_week].iloc[0]
        student_name = student_example['name']
        
        # Get this student's test weeks
        student_test_data = test_data[test_data['name'] == student_name].sort_values('week')
        test_weeks = student_test_data['week'].tolist()
        
        examples.append({
            'student': student_name[:],
            'ends_at': ending_week,
            'test_weeks': test_weeks
        })
    
    for ex in examples:
        print(f"  Student {ex['student']} ends at week {ex['ends_at']} → test weeks: {ex['test_weeks']}")
    
    print()
    print("📈 COMBINED RESULT:")
    test_week_dist = test_data['week'].value_counts().sort_index()
    print("Test samples per week:")
    for week, count in test_week_dist.items():
        print(f"  Week {week:2d}: {count:3d} test samples")
    
    print()
    print(f"📋 SUMMARY:")
    print(f"  • Earliest test week: {test_data['week'].min()}")
    print(f"  • Latest test week: {test_data['week'].max()}")
    print(f"  • Range: {test_data['week'].min()} to {test_data['week'].max()}")
    print(f"  • This is because students have different ending weeks!")
    
    print()
    print("✅ WHY THIS MAKES SENSE:")
    print("  1. Each student gets their PERSONAL last 3 weeks as test data")
    print("  2. Students joined the study at different times")
    print("  3. Some students have longer participation than others")
    print("  4. The 'last 3 weeks' is relative to each student's timeline")
    print("  5. This preserves the chronological split principle per student")

analyze_test_week_range() 

=== WHY TEST WEEKS RANGE FROM 2 TO 8 ===

STEP 3: Split chronologically
Loading tidied data from: ~/cmu/goalsetting-recommendation-algorithm/time-series-predictor/data/data_tidied.csv
Loaded data shape: (1893, 3)

Analyzing data structure for chronological split...
Data analysis for split determination:
  Total students: 184
  Week range: -2 to 8
  Weeks per student - median: 11.0, range: 3-11
  Recommended K: 3
  Reasoning: With median 11.0 weeks per student, K=3 provides ~27.3% test data while ensuring ≥5 training weeks

Splitting data with K = 3 test weeks per student...
Chronological split complete!
  Students processed: 183
  Students with insufficient data (≤3 weeks): 1
  Training samples: 1341
  Test samples: 549
  Training weeks per student: count    183.000000
mean       7.327869
std        1.153894
min        1.000000
25%        7.000000
50%        8.000000
75%        8.000000
max        8.000000
dtype: float64
  Test weeks per student: count    183.0
mean       3.0
std      

## Fake student?! 

Notice that school, teacher, name have the same ID
This is the only student ending at week 0
This is filtered out by K=3 in the `data_processing.py` file

In [5]:
!cd .. && grep "55611e71b358a30158c61810ad802435" exp-static-flexible-anon-2025-05-29.csv

55611e71b358a30158c61810ad802435,55611e71b358a30158c61810ad802435,NA,55611e71b358a30158c61810ad802435,None,0,NA,NA,NA,NA,NA,NA,NA,0,FALSE,NA
55611e71b358a30158c61810ad802435,55611e71b358a30158c61810ad802435,NA,55611e71b358a30158c61810ad802435,None,-1,NA,NA,NA,NA,NA,0,NA,-1,FALSE,NA
55611e71b358a30158c61810ad802435,55611e71b358a30158c61810ad802435,NA,55611e71b358a30158c61810ad802435,None,-2,NA,NA,NA,NA,NA,NA,NA,-2,FALSE,NA
